In [1]:
# After doc extraction, next step is chunking.
# We will look at different chunking methods; but before that let us load one research paper and see the extraction results.

In [2]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

In [3]:
import pandas as pd

In [4]:
from src.extraction import (
    ExtractionContext,
    LayoutExtractor,
    TextExtractor,
    TableExtractor,
    ImageExtractor,
)

/home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
pdf_path = "/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf"

In [7]:
# ── 1. Layout (single Docling pass — must run first) ──────────────────────────
context = ExtractionContext(file_path=pdf_path)

In [8]:
LayoutExtractor().extract(context)          # populates context.layout

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 360/360 [00:00<00:00, 13311.03it/s]
[INFO] 2026-04-01 12:39:56,331 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-01 12:39:56,337 [RapidOCR] download_file.py:60: File exists and is valid: /home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-01 12:39:56,338 [RapidOCR] main.py:53: Using /home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-01 12:39:56,398 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-01 12:39:56,399 [RapidOCR] download_file.py:60: File exists and is valid: /home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-01 12:39:56,400 [RapidOCR] main.py:53: Using /home/pritesh-jha/projects/poc-to-prod/.venv/lib/pyt

[ExtractedRecord(record_type='text', page=1, bbox={'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}, content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023', raw='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 ExtractedRecord(record_type='text', page=1, bbox={'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}, content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', raw='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.'),
 ExtractedRecord(record_type='text', page=1, bbox={'l': 211.488, 't': 641.835626, 'r': 399.89333760000005, 'b': 626.3589814}, content='Attention Is All You Need', raw='Attention Is All You Need'),
 ExtractedRecord(record_type='text', page=1, bbox={'l': 116.68099999999998, 't': 557.6831053999999, 'r': 

In [9]:
# ── 2. Text ───────────────────────────────────────────────────────────────────
texts = TextExtractor().extract(context)

In [10]:
texts[0:5]

[ExtractedRecord(record_type='text', page=1, bbox={'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}, content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023', raw='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 ExtractedRecord(record_type='text', page=1, bbox={'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}, content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', raw='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.'),
 ExtractedRecord(record_type='text', page=1, bbox={'l': 211.488, 't': 641.835626, 'r': 399.89333760000005, 'b': 626.3589814}, content='Attention Is All You Need', raw='Attention Is All You Need'),
 ExtractedRecord(record_type='text', page=1, bbox={'l': 116.68099999999998, 't': 557.6831053999999, 'r': 

In [11]:
for r in texts:
    print(r.page, r.record_type, r.content[:120])

1 text arXiv:1706.03762v7  [cs.CL]  2 Aug 2023
1 text Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this pap
1 text Attention Is All You Need
1 text Ashish Vaswani ∗ Google Brain avaswani@google.com
1 text Noam Shazeer ∗ Google Brain noam@google.com
1 text Llion Jones ∗ Google Research llion@google.com
1 text Niki Parmar ∗ Google Research nikip@google.com
1 text Aidan N. Gomez ∗ † University of Toronto aidan@cs.toronto.edu
1 text Jakob Uszkoreit ∗ Google Research usz@google.com
1 text Łukasz Kaiser ∗ Google Brain lukaszkaiser@google.com
1 text Illia Polosukhin ∗ ‡
1 text illia.polosukhin@gmail.com
1 text Abstract
1 text The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include a
1 text ∗ Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort 
1 text † Work performed while at Google Brain.
1 text ‡ W

In [12]:
# ── 3. Tables (Docling table_cells → DataFrame) ───────────────────────────────
tables = TableExtractor().extract(context)

In [13]:
tables[0:2]

[ExtractedRecord(record_type='table', page=6, bbox={'l': 117.47545623779297, 't': 680.234489440918, 'r': 494.3601989746094, 'b': 604.6219329833984}, content={'dataframe':                     Layer Type Complexity per Layer Sequential Operations  \
 0               Self-Attention        O ( n 2 · d )                 O (1)   
 1                    Recurrent        O ( n · d 2 )               O ( n )   
 2                Convolutional    O ( k · n · d 2 )                 O (1)   
 3  Self-Attention (restricted)      O ( r · n · d )                 O (1)   
 
   Maximum Path Length  
 0               O (1)  
 1             O ( n )  
 2    O ( log k ( n ))  
 3           O ( n/r )  , 'markdown': '| Layer Type                  | Complexity per Layer   | Sequential Operations   | Maximum Path Length   |\n|:----------------------------|:-----------------------|:------------------------|:----------------------|\n| Self-Attention              | O ( n 2 · d )          | O (1)                   | 

In [14]:
for r in tables:
    print(r.page, r.content["dataframe"].shape)
    print(r.content["dataframe"])
    # r.content["markdown"], r.content["csv"], r.content["json"] also available

6 (4, 4)
                    Layer Type Complexity per Layer Sequential Operations  \
0               Self-Attention        O ( n 2 · d )                 O (1)   
1                    Recurrent        O ( n · d 2 )               O ( n )   
2                Convolutional    O ( k · n · d 2 )                 O (1)   
3  Self-Attention (restricted)      O ( r · n · d )                 O (1)   

  Maximum Path Length  
0               O (1)  
1             O ( n )  
2    O ( log k ( n ))  
3           O ( n/r )  
8 (10, 5)
                                    EN-DE  EN-FR          EN-DE          EN-FR
0                     ByteNet [18]  23.75                                     
1           Deep-Att + PosUnk [39]          39.2                 1 . 0 · 10 20
2                   GNMT + RL [38]   24.6  39.92  2 . 3 · 10 19  1 . 4 · 10 20
3                      ConvS2S [9]  25.16  40.46  9 . 6 · 10 18  1 . 5 · 10 20
4                         MoE [32]  26.03  40.56  2 . 0 · 10 19  1 . 2 · 10 20
5

In [15]:
# reconstruct_table takes the raw Docling dict — it's in content["raw_docling"]
table_record = tables[0]                              # first table

In [16]:
def reconstruct_table(table_content: dict) -> pd.DataFrame:
    """
    Reconstruct a pandas DataFrame from Docling's extracted table content.

    Each cell in `data.table_cells` carries its grid position via
    start_row_offset_idx / start_col_offset_idx, so we can place every
    cell into the correct slot of a 2-D grid and then build a DataFrame.
    """
    cells = table_content["data"]["table_cells"]

    # Determine grid dimensions
    num_rows = max(c["end_row_offset_idx"] for c in cells)
    num_cols = max(c["end_col_offset_idx"] for c in cells)

    # Build an empty grid
    grid = [[""] * num_cols for _ in range(num_rows)]

    # Track which rows are purely column-header rows
    header_rows = set()

    for cell in cells:
        r = cell["start_row_offset_idx"]
        c = cell["start_col_offset_idx"]
        grid[r][c] = cell["text"]
        if cell.get("column_header"):
            header_rows.add(r)

    if header_rows:
        # Use the last header row as column names
        header_row_idx = max(header_rows)
        columns = grid[header_row_idx]
        data_rows = [grid[i] for i in range(num_rows) if i not in header_rows]
    else:
        columns = [f"col_{i}" for i in range(num_cols)]
        data_rows = grid

    return pd.DataFrame(data_rows, columns=columns)


In [17]:
table_record = tables[0]

In [18]:
df = reconstruct_table(table_record.content["raw_docling"])

In [19]:
df

,Layer Type,Complexity per Layer,Sequential Operations,Maximum Path Length
0,Self-Attention,O ( n 2 · d ),O (1),O (1)
1,Recurrent,O ( n · d 2 ),O ( n ),O ( n )
2,Convolutional,O ( k · n · d 2 ),O (1),O ( log k ( n ))
3,Self-Attention (restricted),O ( r · n · d ),O (1),O ( n/r )


In [20]:
# ── 4a. Images — crop only (fast, no API call) ────────────────────────────────
extractor = ImageExtractor()
images = extractor.extract(context)

In [21]:
for r in images:
    print(r.page, r.content["classification"])
    with open(f"page_{r.page}_img.png", "wb") as f:
        f.write(r.content["png_bytes"])

3 {'top_class': 'flow_chart', 'predictions': [{'confidence': 0.9998363256454468, 'created_by': 'DocumentPictureClassifier', 'class_name': 'flow_chart'}, {'confidence': 7.412472768919542e-05, 'created_by': 'DocumentPictureClassifier', 'class_name': 'table'}, {'confidence': 3.5353128623683006e-05, 'created_by': 'DocumentPictureClassifier', 'class_name': 'engineering_drawing'}, {'confidence': 1.2677158338192385e-05, 'created_by': 'DocumentPictureClassifier', 'class_name': 'screenshot_from_manual'}, {'confidence': 1.1297190212644637e-05, 'created_by': 'DocumentPictureClassifier', 'class_name': 'other'}, {'confidence': 5.874060661881231e-06, 'created_by': 'DocumentPictureClassifier', 'class_name': 'bar_chart'}, {'confidence': 3.9194665077957325e-06, 'created_by': 'DocumentPictureClassifier', 'class_name': 'screenshot_from_computer'}, {'confidence': 3.8662960832880344e-06, 'created_by': 'DocumentPictureClassifier', 'class_name': 'line_chart'}, {'confidence': 3.0796945793554187e-06, 'created_

In [22]:
from openai import OpenAI

summaries = extractor.process_images(context, OpenAI())


Total images found: 6
  Processing page 3 — type: flow_chart
  Processing page 4 — type: flow_chart
  Processing page 4 — type: flow_chart
  Processing page 13 — type: line_chart
  Processing page 14 — type: line_chart
  Processing page 15 — type: line_chart


In [23]:
for s in summaries:
    print(s["page"], s["type"])
    print(s["description"])
    print(s["key_data_points"])

3 flow_chart
The image visually represents the architecture of a Transformer model, showing the components and flow between an encoder and decoder. It includes elements like input embeddings, positional encodings, multi-head attention, feed forward layers, and output probabilities.
['Output Probabilities', 'Softmax', 'Linear', 'Add & Norm', 'Feed Forward', 'Multi-Head Attention', 'Masked Multi-Head Attention', 'Nx', 'Positional Encoding', 'Input Embedding', 'Output Embedding', 'Inputs', 'Outputs (shifted right)']
4 flow_chart
A flowchart showing the Scaled Dot-Product Attention mechanism with components including MatMul, Scale, Mask (optional), SoftMax, and another MatMul.
['MatMul', 'Scale', 'Mask (opt.)', 'SoftMax', 'Q', 'K', 'V']
4 flow_chart
The image visually represents the architecture of Multi-Head Attention in a neural network, showing the flow from queries (Q), keys (K), and values (V) through linear layers, followed by scaled dot-product attention and concatenation, leading t

In [24]:
# We now know how to extract everythin from a pdf document.
# Now let us look at chunking.
# I have been talking about chunking for a while, but what is it really? Why do we need it? Let's look at the motivation first.

In [25]:
# Let's say we want to build a question-answering system on top of research papers. 
# We have the extracted content, but how do we feed it to an LLM? 
# 
# We can't just dump everything into the prompt, because of the token limit. 
# We need to chunk the content into smaller pieces that can fit into the prompt. 
# But how do we chunk it? Do we chunk it by page? By section? By paragraph? By table? By image?
#  
# There are many ways to chunk the content, and each way has its own pros and cons. 
# Let's explore some of these chunking methods.

In [26]:
# Let us take one very big chunk of text and see how we can chunk it in different ways.

In [27]:
data_directory = "../assets/data/"

In [28]:
documents = ""
with open(file=data_directory+"HRA.txt",mode="r") as f:
    for line in f:
        # print(f.readline())
        documents += f.readline()

In [29]:
len(documents)

12262

In [30]:
# The document is too big to fit into the prompt, so we need to chunk it.

# The first type of chunking is Fixed length chunking with overlap.
# We can use the RecursiveCharacterTextSplitter from langchain to do this.

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [32]:
chunks = splitter.create_documents([documents])

In [33]:
chunks[0:2]

[Document(metadata={}, page_content="From Wikipedia, the free encyclopedia\nHindustan Socialist Republican Association (HSRA), previously known as the Hindustan Republican Army and Hindustan Republican Association (HRA), was a left-wing Indian revolutionary organization, founded by Sachindranath Sanyal. After changes in Bhagat Singh's ideology and the influence of the Russian Revolution, they held meetings in Feroz Shah Kotla Maidan and added the word socialist to their name. Ram Prasad Bismil, Ashfaqulla Khan, Sachindra Nath Bakshi, Sachindranath Sanyal and Jogesh Chandra Chatterjee were the leaders of the group at the time. HSRA's manifesto titled The Revolutionary and written constitution were produced as evidence in the Kakori conspiracy case of 1925. \nOrigins"),
 Document(metadata={}, page_content='Origins\nThe non-cooperation movement of 1919 had led to a large-scale mobilization of the Indian population against the British Raj. Although intended as a non-violent resistance move

In [34]:
len(chunks)

19

In [35]:
# This is a simple but a decent way to chunk the document.
# Howver, it does not take into account the structure of the document.
# For example, it may split a paragraph into two chunks, which may not be ideal for question-answering tasks.
# So we need a hierarchy-aware chunking method that takes into account the structure of the document.
# The next is hierarchy-aware chunking.

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── 1. Two splitters — parent (large) and child (small) ───────────────────────
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

In [37]:
# ── 2. Split text into parent chunks first ────────────────────────────────────
parent_chunks = parent_splitter.create_documents([documents])

In [38]:
len(parent_chunks)

10

In [39]:
# ── 3. Split each parent into children, keeping the parent reference ──────────
all_child_chunks = []

for parent_id, parent in enumerate(parent_chunks):
    children = child_splitter.create_documents(
        [parent.page_content],
        metadatas=[{"parent_id": parent_id, "parent_text": parent.page_content}]
    )
    all_child_chunks.extend(children)


In [40]:
len(all_child_chunks)

51

In [41]:
# So we can see that a document of over 12k characters,
# is now split into 10 parent chunks of 2k characters each (with 200 char overlap),
# and each parent chunk is further split into child chunks of 400 characters each (with 50 char overlap), 
# resulting in a total of 51 child chunks.

In [42]:
# ── 4. Index children, retrieve parent ───────────────────────────────────────
# Small child chunks go into the vector store (for dense retrieval)
# When a child is retrieved, use parent_id to fetch the full parent for the LLM

for chunk in all_child_chunks:
    print(chunk.metadata["parent_id"], chunk.page_content[:80])


0 From Wikipedia, the free encyclopedia
0 Hindustan Socialist Republican Association (HSRA), previously known as the Hindu
0 Maidan and added the word socialist to their name. Ram Prasad Bismil, Ashfaqulla
0 Origins
0 The non-cooperation movement of 1919 had led to a large-scale mobilization of th
0 were locked in a police station, which was subsequently set on fire by demonstra
0 of the Indian National Congress, held in Gaya, Bihar.[4] The political vacuum cr
0 Opposition of Gandhi in Gaya Congress
1 Opposition of Gandhi in Gaya Congress
1 Without ascertaining the facts behind this incident, Mahatma Gandhi, declared an
1 to rescind his decision, the Indian National Congress was divided into two group
2 Sharing responsibility
2 This meeting decided the name of the party would be the Hindustan Republican Ass
2 of the Anushilan Samiti. After attending the meeting in Cawnpore, both Sanyal an
2 The HRA established branches in Agra, Allahabad, Benares, Cawnpore, Lucknow, Sah
2 Early activit

In [43]:
# The research papers are usually structured into sections and subsections.
# Hence, I will be short listing hierarchy chunking into our application.
# But there is one more method that I want to explore and discuss -  The Semantic Chunking.

In [44]:
# Semantic chunking are done using multiple ways. 
# I will explore 2 ways to do it. 

# One of them is to use a text Topic Modeling-Based (e.g. TextTiling) 
# which uses sliding window over sentence blocks, detect topic shifts via vocabulary changes. 
# No embeddings needed, purely lexical.

# Another method is to use an embedding-based approach, 
# where we encode sentences or paragraphs into vector representations and then cluster them based on semantic similarity.

In [45]:
import re
from collections import Counter
from typing import Any, List, Optional
 
from langchain_core.documents import Document
from langchain_text_splitters import TextSplitter

In [46]:
def _tokenize(text: str) -> List[str]:
    """Lowercase, strip punctuation, return tokens."""
    return re.findall(r"[a-z]+", text.lower())
 

In [47]:
def _sentence_tokenize(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

In [48]:
def _block_score(block_a: List[List[str]], block_b: List[List[str]]) -> float:
    """
    Cosine similarity between two token blocks using bag-of-words vectors.
    Each block is a list of sentence token lists.
    """
    counter_a: Counter = Counter(t for sent in block_a for t in sent)
    counter_b: Counter = Counter(t for sent in block_b for t in sent)
 
    vocab = set(counter_a) | set(counter_b)
    dot = sum(counter_a[w] * counter_b[w] for w in vocab)
    norm_a = sum(v ** 2 for v in counter_a.values()) ** 0.5
    norm_b = sum(v ** 2 for v in counter_b.values()) ** 0.5
 
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

In [49]:
def _depth_score(scores: List[float], i: int) -> float:
    """
    Depth score at position i: how much the score dips relative to
    its left and right peaks. Captures valley depth.
    """
    left_peak = scores[i]
    for j in range(i - 1, -1, -1):
        if scores[j] >= left_peak:
            left_peak = scores[j]
        else:
            break
 
    right_peak = scores[i]
    for j in range(i + 1, len(scores)):
        if scores[j] >= right_peak:
            right_peak = scores[j]
        else:
            break
 
    return (left_peak - scores[i]) + (right_peak - scores[i])

In [50]:
class TextTilingSemanticChunker(TextSplitter):
    """
    Lexical semantic chunker based on the TextTiling algorithm.
    No embeddings or external models required.
 
    Algorithm:
        1. Split text into sentences and tokenize each sentence.
        2. Group sentences into overlapping windows of size `window_size`.
        3. Compute lexical similarity (bag-of-words cosine) between adjacent
           windows at every sentence boundary (gap score).
        4. Convert gap scores to depth scores (valley depth).
        5. Insert chunk boundaries at depth scores above `depth_threshold`.
 
    Args:
        window_size: Number of sentences on each side of a boundary gap
                     used to compute the block similarity score.
        depth_threshold: Depth score above which a boundary is inserted.
                         Higher = fewer, larger chunks.
        min_chunk_sentences: Minimum number of sentences to keep in a chunk
                             before a split is allowed.
        **kwargs: Forwarded to langchain TextSplitter base class.
    """
 
    def __init__(
        self,
        window_size: int = 3,
        depth_threshold: float = 0.3,
        min_chunk_sentences: int = 2,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self._window_size = window_size
        self._depth_threshold = depth_threshold
        self._min_chunk_sentences = min_chunk_sentences
 
    # ------------------------------------------------------------------
    # LangChain TextSplitter interface
    # ------------------------------------------------------------------
 
    def split_text(self, text: str) -> List[str]:
        sentences = _sentence_tokenize(text)
        if len(sentences) <= 1:
            return sentences
 
        token_seqs = [_tokenize(s) for s in sentences]
        n = len(sentences)
        w = self._window_size
 
        # Gap scores at each boundary i (between sentence i and i+1)
        gap_scores: List[float] = []
        for i in range(n - 1):
            left_start = max(0, i - w + 1)
            right_end = min(n, i + w + 1)
            block_a = token_seqs[left_start : i + 1]
            block_b = token_seqs[i + 1 : right_end]
            gap_scores.append(_block_score(block_a, block_b))
 
        depth_scores = [_depth_score(gap_scores, i) for i in range(len(gap_scores))]
 
        boundaries: List[int] = []
        for i, d in enumerate(depth_scores):
            if d >= self._depth_threshold:
                # Enforce min chunk size from previous boundary
                prev = boundaries[-1] if boundaries else -1
                if (i - prev) >= self._min_chunk_sentences:
                    boundaries.append(i)
 
        chunks: List[str] = []
        prev_idx = 0
        for b in boundaries:
            chunk_sents = sentences[prev_idx : b + 1]
            chunks.append(" ".join(chunk_sents))
            prev_idx = b + 1
 
        if prev_idx < n:
            chunks.append(" ".join(sentences[prev_idx:]))
 
        return [c for c in chunks if c.strip()]
 
    def create_documents(
        self,
        texts: List[str],
        metadatas: Optional[List[dict]] = None,
    ) -> List[Document]:
        metadatas = metadatas or [{} for _ in texts]
        documents: List[Document] = []
        for text, meta in zip(texts, metadatas):
            for chunk in self.split_text(text):
                documents.append(Document(page_content=chunk, metadata=dict(meta)))
        return documents
 
    def split_documents(self, documents: List[Document]) -> List[Document]:
        texts = [doc.page_content for doc in documents]
        metas = [doc.metadata for doc in documents]
        return self.create_documents(texts, metas)

In [51]:
chunker = TextTilingSemanticChunker(
    window_size=3,
    depth_threshold=0.3,
    min_chunk_sentences=2,
)

In [52]:
chunks = chunker.split_text(documents)

In [53]:
chunks[0:2]

["From Wikipedia, the free encyclopedia\nHindustan Socialist Republican Association (HSRA), previously known as the Hindustan Republican Army and Hindustan Republican Association (HRA), was a left-wing Indian revolutionary organization, founded by Sachindranath Sanyal. After changes in Bhagat Singh's ideology and the influence of the Russian Revolution, they held meetings in Feroz Shah Kotla Maidan and added the word socialist to their name. Ram Prasad Bismil, Ashfaqulla Khan, Sachindra Nath Bakshi, Sachindranath Sanyal and Jogesh Chandra Chatterjee were the leaders of the group at the time. HSRA's manifesto titled The Revolutionary and written constitution were produced as evidence in the Kakori conspiracy case of 1925. Origins\nThe non-cooperation movement of 1919 had led to a large-scale mobilization of the Indian population against the British Raj. Although intended as a non-violent resistance movement, due to heightened tensions, and a brutal response by the British police forces,

In [54]:
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i + 1}]\n{chunk}\n")

[Chunk 1]
From Wikipedia, the free encyclopedia
Hindustan Socialist Republican Association (HSRA), previously known as the Hindustan Republican Army and Hindustan Republican Association (HRA), was a left-wing Indian revolutionary organization, founded by Sachindranath Sanyal. After changes in Bhagat Singh's ideology and the influence of the Russian Revolution, they held meetings in Feroz Shah Kotla Maidan and added the word socialist to their name. Ram Prasad Bismil, Ashfaqulla Khan, Sachindra Nath Bakshi, Sachindranath Sanyal and Jogesh Chandra Chatterjee were the leaders of the group at the time. HSRA's manifesto titled The Revolutionary and written constitution were produced as evidence in the Kakori conspiracy case of 1925. Origins
The non-cooperation movement of 1919 had led to a large-scale mobilization of the Indian population against the British Raj. Although intended as a non-violent resistance movement, due to heightened tensions, and a brutal response by the British police f

In [55]:
# As we can see the output is quite good. 
# The chunker is able to identify the different sections of the document and split it accordingly.
# Also, the latency is quite low, since it does not require any external API calls or embeddings.

In [56]:
# Now let us try the embedding-based semantic chunking method.

In [57]:

from typing import Any, List, Optional
 
import numpy as np
from langchain_core.documents import Document
from langchain_text_splitters import TextSplitter
from sentence_transformers import SentenceTransformer

In [58]:
def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

In [59]:
def _split_into_sentences(text: str) -> List[str]:
    import re
    sentence_endings = re.compile(r"(?<=[.!?])\s+")
    sentences = sentence_endings.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]

In [60]:
class EmbeddingSemanticChunker(TextSplitter):
    """
    Splits text into semantically coherent chunks using sentence-level
    embedding cosine similarity.

    Algorithm:
        1. Split text into sentences.
        2. Embed each sentence with a SentenceTransformer model.
        3. Compute cosine similarity between consecutive sentence embeddings.
        4. Insert a chunk boundary wherever similarity drops below `threshold`.
        5. Merge consecutive sentences within each chunk.

    Args:
        model_name: Any sentence-transformers model name.
        threshold: Cosine similarity below which a boundary is inserted.
                   Range [0, 1]. Lower values = fewer, larger chunks.
        min_chunk_size: Minimum number of sentences per chunk. Prevents
                        single-sentence fragments when threshold is tight.
        batch_size: Batch size passed to SentenceTransformer.encode().
        **kwargs: Forwarded to langchain TextSplitter base class.
    """

    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        threshold: float = 0.5,
        min_chunk_size: int = 2,
        batch_size: int = 64,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self._model = SentenceTransformer(model_name)
        self._threshold = threshold
        self._min_chunk_size = min_chunk_size
        self._batch_size = batch_size

    # ------------------------------------------------------------------
    # LangChain TextSplitter interface
    # ------------------------------------------------------------------

    def split_text(self, text: str) -> List[str]:
        sentences = _split_into_sentences(text)
        if not sentences:
            return []
        if len(sentences) == 1:
            return sentences

        embeddings: np.ndarray = self._model.encode(
            sentences,
            batch_size=self._batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

        similarities = [
            _cosine_similarity(embeddings[i], embeddings[i + 1])
            for i in range(len(embeddings) - 1)
        ]

        chunks: List[str] = []
        current: List[str] = [sentences[0]]

        for i, sim in enumerate(similarities):
            next_sentence = sentences[i + 1]
            if sim < self._threshold and len(current) >= self._min_chunk_size:
                chunks.append(" ".join(current))
                current = [next_sentence]
            else:
                current.append(next_sentence)

        if current:
            chunks.append(" ".join(current))

        return chunks

    def create_documents(
        self,
        texts: List[str],
        metadatas: Optional[List[dict]] = None,
    ) -> List[Document]:
        metadatas = metadatas or [{} for _ in texts]
        documents: List[Document] = []
        for text, meta in zip(texts, metadatas):
            for chunk in self.split_text(text):
                documents.append(Document(page_content=chunk, metadata=dict(meta)))
        return documents

    def split_documents(self, documents: List[Document]) -> List[Document]:
        texts = [doc.page_content for doc in documents]
        metas = [doc.metadata for doc in documents]
        return self.create_documents(texts, metas)


In [61]:
chunker = EmbeddingSemanticChunker(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    threshold=0.5,
    min_chunk_size=2,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5904.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
chunks = chunker.split_text(documents)

In [63]:
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i + 1}]\n{chunk}\n")

[Chunk 1]
From Wikipedia, the free encyclopedia
Hindustan Socialist Republican Association (HSRA), previously known as the Hindustan Republican Army and Hindustan Republican Association (HRA), was a left-wing Indian revolutionary organization, founded by Sachindranath Sanyal. After changes in Bhagat Singh's ideology and the influence of the Russian Revolution, they held meetings in Feroz Shah Kotla Maidan and added the word socialist to their name.

[Chunk 2]
Ram Prasad Bismil, Ashfaqulla Khan, Sachindra Nath Bakshi, Sachindranath Sanyal and Jogesh Chandra Chatterjee were the leaders of the group at the time. HSRA's manifesto titled The Revolutionary and written constitution were produced as evidence in the Kakori conspiracy case of 1925.

[Chunk 3]
Origins
The non-cooperation movement of 1919 had led to a large-scale mobilization of the Indian population against the British Raj. Although intended as a non-violent resistance movement, due to heightened tensions, and a brutal response b